# Python Automation

Automating repetitive tasks with the standard library.

**Install:** `pip install schedule watchdog`

**In this notebook:**
- pathlib — file system operations
- shutil — copy, move, delete
- CSV and JSON automation
- subprocess — run shell commands
- os.walk — directory traversal
- schedule — task scheduling
- logging — structured output
- File organiser pattern

## 1. pathlib — File System Operations

In [ ]:
from pathlib import Path

# Navigation
print(Path.cwd())
print(list(Path('.').glob('*.py'))[:3])
print(list(Path('.').rglob('*.py'))[:3])   # recursive

# File properties
p = Path('README.md')
if p.exists():
    print(p.name, p.suffix, p.stem, p.parent)
    print(f'{p.stat().st_size} bytes')

# Create directory tree
Path('tmp/a/b').mkdir(parents=True, exist_ok=True)
Path('tmp/a/b/file.txt').write_text('hello', encoding='utf-8')
print(Path('tmp/a/b/file.txt').read_text())

import shutil; shutil.rmtree('tmp', ignore_errors=True)

## 2. shutil — Copy, Move, Delete

In [ ]:
from pathlib import Path
import shutil

# Setup
Path('demo').mkdir(exist_ok=True)
Path('demo/original.txt').write_text('content', encoding='utf-8')

# Copy
shutil.copy('demo/original.txt', 'demo/copy.txt')       # without metadata
shutil.copy2('demo/original.txt', 'demo/copy2.txt')     # with metadata

# Move
Path('demo/dest').mkdir(exist_ok=True)
shutil.move('demo/copy.txt', 'demo/dest/copy.txt')

# Copy entire tree
shutil.copytree('demo', 'demo_backup')

# Delete
shutil.rmtree('demo', ignore_errors=True)
shutil.rmtree('demo_backup', ignore_errors=True)
print('Done')

## 3. CSV and JSON Automation

In [ ]:
import csv, json
from pathlib import Path

# CSV write → read
students = [{'name': 'Alice', 'score': 95}, {'name': 'Bob', 'score': 87}]
with open('students.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['name', 'score'])
    w.writeheader(); w.writerows(students)

with open('students.csv', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(rows)

# JSON write → read → update
cfg = {'version': '1.0', 'debug': False}
Path('config.json').write_text(json.dumps(cfg, indent=2))
data = json.loads(Path('config.json').read_text())
data['version'] = '2.0'
Path('config.json').write_text(json.dumps(data, indent=2))
print(json.loads(Path('config.json').read_text()))

Path('students.csv').unlink(missing_ok=True)
Path('config.json').unlink(missing_ok=True)

## 4. subprocess — Run Shell Commands

In [ ]:
import subprocess

# Run command and capture output
r = subprocess.run(['python', '--version'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)      # Python 3.x.x
print('Return code:', r.returncode)

# check=True raises CalledProcessError on non-zero exit
try:
    subprocess.run(['python', '-c', 'import sys; sys.exit(1)'], check=True)
except subprocess.CalledProcessError as e:
    print('Failed with code:', e.returncode)

## 5. os.walk — Directory Traversal

In [ ]:
import os
from pathlib import Path
import tempfile

# Create a temp tree
with tempfile.TemporaryDirectory() as tmp:
    for i in range(2):
        d = Path(tmp) / f'sub{i}'
        d.mkdir()
        (d / f'file{i}.txt').write_text('data')
    (Path(tmp) / 'root.py').write_text('code')

    total = 0
    for dirpath, dirnames, filenames in os.walk(tmp):
        for fname in filenames:
            fpath = os.path.join(dirpath, fname)
            size  = os.path.getsize(fpath)
            total += size
            print(f'{fpath}  ({size} bytes)')
    print(f'Total: {total} bytes')

## 6. Logging

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[logging.StreamHandler()]
)

logger = logging.getLogger('automation')
logger.debug('Debug: detailed info')
logger.info('Info: task started')
logger.warning('Warning: disk 90%% full')
logger.error('Error: file not found')
logger.critical('Critical: system failure')

## 7. File Organiser Pattern

In [ ]:
from pathlib import Path
import shutil, tempfile

EXT_MAP = {
    '.pdf': 'documents', '.txt': 'documents',
    '.png': 'images',    '.jpg': 'images',
    '.py':  'scripts',   '.csv': 'data',
}

def organise(folder):
    src = Path(folder)
    counts = {}
    for file in src.iterdir():
        if not file.is_file():
            continue
        category = EXT_MAP.get(file.suffix.lower(), 'misc')
        dest = src / category
        dest.mkdir(exist_ok=True)
        shutil.move(str(file), dest / file.name)
        counts[category] = counts.get(category, 0) + 1
        print(f'  {file.name} → {category}/')
    return counts

with tempfile.TemporaryDirectory() as tmp:
    for name in ['report.pdf', 'photo.png', 'data.csv', 'app.py', 'unknown.xyz']:
        (Path(tmp) / name).write_text('content')
    print(organise(tmp))

## 8. Task Scheduling with schedule

```python
import schedule, time

def daily_report():
    print('Generating report...')

schedule.every().day.at('09:00').do(daily_report)
schedule.every(30).minutes.do(lambda: print('Ping!'))

while True:
    schedule.run_pending()
    time.sleep(60)
```

> Run this as a standalone script (`python scheduler.py`) — it blocks the process.

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | pathlib, shutil, CSV/JSON |
| [02-medium.py](exercises/02-medium.py) | Medium | File organiser, subprocess, logging |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Backup, sync, report generator |

Solutions: [solutions/](solutions/)